# Protein–Lipid Interaction Analysis

Quantifies how the lipids of the surrounding membrane interact with an ion
channel: how close each lipid species gets, and which residues of the channel
they contact.

| Output | What it measures |
|---|---|
| Minimum lipid–protein distance | Per lipid molecule, per frame: the closest approach to the protein |
| Distance distributions | Histogram and violin plot per lipid species |
| Residue contact probability | Fraction of frames each residue is within the contact cutoff of a given lipid species |
| Enrichment summary | Which species sit closest, and which residues each one prefers |

The point of the residue-contact profile is to find **lipid binding sites**:
positions where a particular species is in contact far more often than the
membrane average, which is what a specific, structured interaction looks like.

---

## Two distances, two questions

The notebook computes two different things, and they answer different questions:

- **`lipid_distance`** gives, for every lipid molecule and frame, its minimum
  distance to the protein. Pooled over molecules and frames this is a
  *distribution*: how tightly does this species pack against the channel?
- **`lipid_residue_contacts`** gives a boolean per (frame, residue, lipid):
  is this residue within the cutoff of this lipid? Averaged, this is a
  *per-residue probability*: where on the channel does this species bind?

Both are computed with periodic boundary conditions applied.

## Expected data layout

```
<DATA_ROOT>/
├── coarse/
│   ├── 310K15/
│   │   ├── NPT.gro
│   │   └── NPT.xtc
│   └── ...
└── atomistic/
    └── ...
```

Note this notebook needs the **full system** — protein plus membrane — not a
protein-only trajectory, so it reads `NPT.gro` / `NPT.xtc` rather than the
protein-centred files used elsewhere in this repository.

## Generating the input files

```bash
# Keep the whole system, make molecules whole, centre on the protein
gmx trjconv -s npt.tpr -f npt.xtc -o NPT.xtc -pbc mol -center
gmx trjconv -s npt.tpr -f NPT.xtc -o NPT.gro -dump 0
```

Do **not** strip the lipids, and do not use `-pbc atom`, which would break
molecules across the boundary and corrupt the minimum-distance calculation.

## Contact cutoffs are resolution-dependent

A coarse-grained bead represents several heavy atoms, so its contact distance is
larger than an atomistic one. The defaults are 6.0 Å for Martini-style
coarse-grained beads and 4.5 Å for all-atom heavy atoms; both are set per model
level in the configuration cell. Hydrogens are excluded in the atomistic case,
which is why the atomistic cutoff refers to heavy atoms only.

## Units

Distances are reported in **ångström** throughout, which is what MDAnalysis
returns. Note this differs from GROMACS' own nm convention.

## Adapting to a different system — checklist

1. `LIPID_RESNAMES` — the resnames of each species, per model level. These differ
   between force fields (`CHOL` in Martini vs `CHL1` in CHARMM, `DPSM` vs `SSM`).
2. `CONTACT_THRESHOLD` — per model level, as above.
3. `PROTEIN_SELECTION` — per model level. `"protein"` works when resnames are
   standard; coarse-grained models often need an explicit selection.
4. `N_PROTOMERS`, `RESIDUE_OFFSET`, `DOMAIN_SEGMENTS` — as in the other notebooks.
5. `ANALYSIS_CONDITIONS` — which grid points to analyse. This is the expensive
   notebook, so it defaults to a single condition rather than all 18.


---

## 1. Imports

In [ ]:
# ---------------------------------------------------------------------------
# Standard library
# ---------------------------------------------------------------------------
import os
import re
import warnings
import itertools
from dataclasses import dataclass
from pathlib import Path

# ---------------------------------------------------------------------------
# Third party
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.patches import Patch                 # domain legend handles
from matplotlib.ticker import ScalarFormatter        # compact axis labels
import seaborn as sns                                # violin plots

import MDAnalysis as mda
from MDAnalysis.lib.distances import distance_array, capped_distance
from scipy.spatial import cKDTree                    # fast nearest-neighbour queries

# MDAnalysis warns about topology attributes that a .gro file legitimately does
# not carry (masses, elements). Silence only that, rather than filtering all
# warnings globally - a blanket filter also hides real numerical problems.
warnings.filterwarnings("ignore", category=UserWarning, module="MDAnalysis")

# ---------------------------------------------------------------------------
# Shared repository configuration
# ---------------------------------------------------------------------------
# channel_config.py sits beside this notebook and holds everything common to
# every notebook here: the data root, the simulation grid, the channel topology,
# the domain boundaries and the plotting primitives. The star import brings in
# only the shared, read-only names it lists in __all__.
import channel_config as cc
from channel_config import *

%matplotlib inline

print(f"numpy {np.__version__}")
print(f"pandas {pd.__version__}")
print(f"MDAnalysis {mda.__version__}")
print(f"seaborn {sns.__version__}")

---

## 2. Configuration

Everything shared with the other notebooks lives in `channel_config.py`; only this notebook's own settings are here.

In [ ]:
# ===========================================================================
#                            C O N F I G U R A T I O N
#
#  Only the settings specific to THIS notebook live here. Everything shared
#  across the repository - the data root, the simulation grid, the channel
#  topology, the domain boundaries and the plot style - lives in
#  channel_config.py, so it can only ever be defined once.
#
#  To change a shared value (protomer count, temperatures, domains, ...), edit
#  channel_config.py and every notebook picks it up.
# ===========================================================================

# ---------------------------------------------------------------------------
# 1. This notebook
# ---------------------------------------------------------------------------
# Set USE_DEMO_DATA = True to generate small synthetic inputs and run the whole
# notebook without real trajectories. Set it to False to use your own data;
# point channel_config.REAL_DATA_ROOT at it, or set the MD_DATA_ROOT
# environment variable.
USE_DEMO_DATA = True

# Names this notebook's output directory, results/<NOTEBOOK_SLUG>/. Change it if
# you copy this notebook as the basis for a new analysis, so the two cannot
# overwrite each other's figures.
NOTEBOOK_SLUG = "protein_lipid_network"

# Write every figure to OUTPUT_DIR as it is drawn.
SAVE_FIGURES = False

# Logical name -> file name inside each condition directory.
FILENAMES = {
    # The FULL system, protein plus membrane - not a protein-only trajectory.
    "topology": "NPT.gro",
    "trajectory": "NPT.xtc",
}

# Resolved paths.
DATA_ROOT = cc.resolve_data_root(USE_DEMO_DATA)
DEMO_DATA_ROOT = cc.DEMO_DATA_ROOT
OUTPUT_DIR = cc.make_output_dir(NOTEBOOK_SLUG)


def build_path(model_level, temperature_k, salinity_mm, key):
    """Return the full path to one input file.

    Override this if your directory tree is shaped differently, e.g. if the
    model level is a filename suffix rather than a parent directory. The
    condition directory name itself comes from cc.condition_dirname().
    """
    return (
        DATA_ROOT
        / model_level
        / cc.condition_dirname(temperature_k, salinity_mm)
        / FILENAMES[key]
    )


def save_figure(fig, name):
    """Save a figure into this notebook's output directory."""
    return cc.save_figure(fig, name, OUTPUT_DIR, SAVE_FIGURES)


# ---------------------------------------------------------------------------
# 2. Lipid species and selections
# ---------------------------------------------------------------------------
# Resnames per model level. Force fields disagree on lipid naming, so these are
# kept separate and keyed by a common display label.
#
# The keys of LIPID_RESNAMES are the labels used in plots and tables; the values
# are the resname to select at each model level.
LIPID_RESNAMES = {
    # label     coarse    atomistic
    "CHOL":   {"coarse": "CHOL", "atomistic": "CHL1"},
    "POPC":   {"coarse": "POPC", "atomistic": "POPC"},
    "DLPE":   {"coarse": "DLPE", "atomistic": "DLPE"},
    "POPE":   {"coarse": "POPE", "atomistic": "POPE"},
    "PAPC":   {"coarse": "PAPC", "atomistic": "PAPC"},
    "PAPS":   {"coarse": "PAPS", "atomistic": "PAPS"},
    "SM":     {"coarse": "DPSM", "atomistic": "SSM"},
}

# Colour per lipid label, used consistently across every figure.
LIPID_COLORS = {
    "CHOL": "#FDE725", "POPC": "#90D743", "DLPE": "#35B779", "POPE": "#21918C",
    "PAPC": "#31688E", "PAPS": "#443983", "SM": "#440154",
}

# How the protein is selected at each model level. Coarse-grained models often
# fall outside MDAnalysis' notion of "protein", so an explicit selection may be
# needed - e.g. "name BB SC1 SC2 SC3 SC4" for Martini.
PROTEIN_SELECTION = {
    "coarse": "protein",
    "atomistic": "protein",
}

# Contact cutoff (A) per model level. A coarse bead stands for several heavy
# atoms, so it needs a larger cutoff than an all-atom heavy atom does.
CONTACT_THRESHOLD = {
    "coarse": 6.0,
    "atomistic": 4.5,
}

# Exclude hydrogens? Meaningless for a coarse-grained model, which has none.
HEAVY_ATOMS_ONLY = {
    "coarse": False,
    "atomistic": True,
}

# ---------------------------------------------------------------------------
# 3. Frame selection and cost control
# ---------------------------------------------------------------------------
# This notebook is the expensive one: cost scales as
#   (frames) x (lipid molecules) x (protein atoms)
# so it analyses ONE condition by default rather than the whole grid. Add more
# (temperature, salinity) pairs once you know how long one takes.
ANALYSIS_CONDITIONS = [
    {"temperature_k": 310, "salinity_mm": 150},
]

# The demo trajectories are only a few dozen frames long, so a stride meant for
# a real run would leave nothing to analyse.
FRAME_STEP = 1 if USE_DEMO_DATA else 50
START_FRAME = 0
END_FRAME = None          # None -> to the end of each trajectory

# Cap on the minimum distance (A). Pairs beyond this are recorded as infinity,
# which speeds up the neighbour search considerably. None -> no cap, every
# distance is exact. Set this only if you care solely about close contacts.
MAX_DISTANCE = None


# ---------------------------------------------------------------------------
# Apply the shared plot style and report the resolved configuration
# ---------------------------------------------------------------------------
cc.apply_style()
cc.describe_config(DATA_ROOT, USE_DEMO_DATA, OUTPUT_DIR)

---

## 3. The condition grid

In [ ]:
# The set of simulations, built once from the shared grid definition. Every
# later selection is a query against it - grid.where(salinity_mm=50) returns the
# positions of those runs - so slicing is never hard coded as [0:6] or [12:18].
grid = cc.ConditionGrid(TEMPERATURES_K, SALINITIES_MM, build_path)
grid.describe()

# Resolve ANALYSIS_CONDITIONS into grid indices once, so the analysis cells and
# the plotting cells cannot disagree about which runs were used.
analysis_indices = []
for entry in ANALYSIS_CONDITIONS:
    analysis_indices.extend(grid.where(**entry))
analysis_indices = sorted(set(analysis_indices))

print(f"\nAnalysing {len(analysis_indices)} condition(s):")
for index in analysis_indices:
    print(f"  [{index:2d}] {grid[index]}")

---

## 4. Demo data (optional)

In [ ]:
# ===========================================================================
#  Optional: synthetic demo data
#
#  Builds a small protein-plus-membrane system and writes it as .gro/.xtc, so
#  the whole notebook runs without real trajectories.
#
#  A handful of residues are given an artificial affinity for cholesterol, so
#  that the residue-contact profile has a real binding site to find rather than
#  uniform noise. Everything here is fabricated.
# ===========================================================================

DEMO_RESIDUES_PER_PROTOMER = 60      # small, so the demo runs in seconds
DEMO_ATOMS_PER_RESIDUE = 2
DEMO_MOLECULES_PER_LIPID = 8
DEMO_ATOMS_PER_LIPID = 4
DEMO_N_FRAMES = 12
DEMO_BOX = 110.0                     # A, cubic
DEMO_PROTEIN_RADIUS = 18.0           # A, radius of the protein cylinder
DEMO_BINDING_RESIDUES = (12, 13, 14, 40, 41)   # 0-based, per protomer


def _build_demo_universe(model_level):
    """Assemble an MDAnalysis Universe: a protein cylinder in a lipid bath.

    Residue numbering is chosen so that adding RESIDUE_OFFSET[model_level] puts
    every model level onto the common numbering scheme, exactly as real data
    from the two resolutions would.
    """
    n_res_protein = DEMO_RESIDUES_PER_PROTOMER * N_PROTOMERS
    n_atoms_protein = n_res_protein * DEMO_ATOMS_PER_RESIDUE

    lipid_labels = list(LIPID_RESNAMES)
    n_res_lipid = len(lipid_labels) * DEMO_MOLECULES_PER_LIPID
    n_atoms_lipid = n_res_lipid * DEMO_ATOMS_PER_LIPID

    n_residues = n_res_protein + n_res_lipid
    n_atoms = n_atoms_protein + n_atoms_lipid

    # Protein residues use a standard amino-acid resname so that the "protein"
    # selection works; lipids use the resname for this model level.
    resnames = ["ALA"] * n_res_protein
    for label in lipid_labels:
        resnames += [LIPID_RESNAMES[label][model_level]] * DEMO_MOLECULES_PER_LIPID

    atom_resindex = np.concatenate([
        np.repeat(np.arange(n_res_protein), DEMO_ATOMS_PER_RESIDUE),
        np.repeat(np.arange(n_res_protein, n_residues), DEMO_ATOMS_PER_LIPID),
    ])

    names = (["BB", "SC1"] * n_res_protein
             + ["GL1", "GL2", "C1A", "C1B"] * n_res_lipid)

    universe = mda.Universe.empty(
        n_atoms, n_residues=n_residues, atom_resindex=atom_resindex,
        trajectory=True)
    universe.add_TopologyAttr("name", names)
    universe.add_TopologyAttr("type", [n[0] for n in names])
    universe.add_TopologyAttr("resname", resnames)
    first_residue = RESIDUE_NUMBER_ORIGIN - RESIDUE_OFFSET.get(model_level, 0)
    universe.add_TopologyAttr(
        "resid",
        list(range(first_residue, first_residue + n_res_protein))
        + list(range(5000, 5000 + n_res_lipid)))
    universe.add_TopologyAttr("segid", ["SYS"])
    universe.dimensions = [DEMO_BOX] * 3 + [90.0] * 3
    return universe, n_atoms_protein, n_res_protein, lipid_labels


def _demo_positions(seed, n_atoms_protein, n_res_protein, lipid_labels):
    """Positions for one frame: protein on a cylinder, lipids around it."""
    rng = np.random.default_rng(seed)
    centre = DEMO_BOX / 2.0

    # --- protein: a cylinder of residues, so it has a defined surface --------
    n_res_per_prot = DEMO_RESIDUES_PER_PROTOMER
    protein = np.empty((n_atoms_protein, 3))
    index = 0
    for protomer in range(N_PROTOMERS):
        base_angle = protomer * 2.0 * np.pi / N_PROTOMERS
        for residue in range(n_res_per_prot):
            angle = base_angle + 0.05 * residue
            z = centre + (residue - n_res_per_prot / 2.0) * 0.9
            for atom in range(DEMO_ATOMS_PER_RESIDUE):
                radius = DEMO_PROTEIN_RADIUS + 1.5 * atom
                protein[index] = [
                    centre + radius * np.cos(angle) + rng.normal(0, 0.3),
                    centre + radius * np.sin(angle) + rng.normal(0, 0.3),
                    z + rng.normal(0, 0.3),
                ]
                index += 1

    # --- lipids -------------------------------------------------------------
    lipid_chunks = []
    for label in lipid_labels:
        for molecule in range(DEMO_MOLECULES_PER_LIPID):
            # Cholesterol is placed preferentially next to the binding residues,
            # so the contact profile has a genuine site to recover.
            if label == "CHOL" and molecule < DEMO_MOLECULES_PER_LIPID // 2:
                residue = DEMO_BINDING_RESIDUES[molecule % len(DEMO_BINDING_RESIDUES)]
                protomer = molecule % N_PROTOMERS
                angle = protomer * 2.0 * np.pi / N_PROTOMERS + 0.05 * residue
                z = centre + (residue - n_res_per_prot / 2.0) * 0.9
                radius = DEMO_PROTEIN_RADIUS + rng.uniform(3.0, 6.0)
                anchor = np.array([centre + radius * np.cos(angle),
                                   centre + radius * np.sin(angle), z])
            else:
                angle = rng.uniform(0, 2 * np.pi)
                radius = DEMO_PROTEIN_RADIUS + rng.uniform(5.0, 30.0)
                anchor = np.array([centre + radius * np.cos(angle),
                                   centre + radius * np.sin(angle),
                                   rng.uniform(0, DEMO_BOX)])
            lipid_chunks.append(anchor + rng.normal(0, 1.5,
                                                    (DEMO_ATOMS_PER_LIPID, 3)))

    return np.vstack([protein] + lipid_chunks) % DEMO_BOX


def generate_demo_dataset():
    """Write a .gro/.xtc pair for every condition being analysed."""
    written = 0
    for model_level in MODEL_LEVELS:
        universe, n_atoms_protein, n_res_protein, lipid_labels = \
            _build_demo_universe(model_level)
        for index in analysis_indices:
            cond = grid[index]
            topology = build_path(model_level, cond.temperature_k,
                                  cond.salinity_mm, "topology")
            trajectory = build_path(model_level, cond.temperature_k,
                                    cond.salinity_mm, "trajectory")
            topology.parent.mkdir(parents=True, exist_ok=True)

            universe.atoms.positions = _demo_positions(
                0, n_atoms_protein, n_res_protein, lipid_labels)
            universe.atoms.write(str(topology))

            with mda.Writer(str(trajectory), n_atoms=len(universe.atoms)) as writer:
                for frame in range(DEMO_N_FRAMES):
                    universe.atoms.positions = _demo_positions(
                        frame, n_atoms_protein, n_res_protein, lipid_labels)
                    writer.write(universe.atoms)
            written += 2
    return written


if USE_DEMO_DATA:
    n_written = generate_demo_dataset()
    print(f"Wrote {n_written} synthetic files under {DEMO_DATA_ROOT.resolve()}")
    print(f"System: {DEMO_RESIDUES_PER_PROTOMER} residues x {N_PROTOMERS} protomers, "
          f"{len(LIPID_RESNAMES)} lipid species x {DEMO_MOLECULES_PER_LIPID} molecules, "
          f"{DEMO_N_FRAMES} frames")
    print("Cholesterol is artificially enriched near residues "
          f"{[r + RESIDUE_NUMBER_ORIGIN for r in DEMO_BINDING_RESIDUES]} "
          "so the contact profile has a site to find.")
    print("\nThese files are for testing the pipeline only - not simulation data.")
else:
    for model_level in MODEL_LEVELS:
        for key in ("topology", "trajectory"):
            missing = [p for p in grid.paths(model_level, key, analysis_indices)
                       if not p.is_file()]
            if missing:
                print(f"  WARNING {model_level}/{key}: {len(missing)} file(s) missing, "
                      f"first is {missing[0]}")

---

## 5. Lipid analysis

In [ ]:
# ===========================================================================
#  Lipid-protein interaction analysis
#
#  Two quantities, computed over the same trajectory:
#
#    lipid_distance()          per (frame, lipid molecule) minimum distance to
#                              the protein - a distribution of closest approach
#    lipid_residue_contacts()  per (frame, protein residue, lipid molecule)
#                              boolean within-cutoff flag - a binding-site map
#
#  Both apply periodic boundary conditions. The minimum-distance routine picks
#  its algorithm from the box geometry: a KD-tree for orthorhombic boxes (fast,
#  and handles the periodic images directly) and MDAnalysis' distance_array in
#  chunks for triclinic boxes, where the KD-tree cannot represent the periodicity.
# ===========================================================================

class LipidAnalysis:
    """Lipid-protein distances and contacts for one or more simulations.

    Parameters
    ----------
    topologies, trajectories : sequence of path
        Matched lists. Must contain the full system, protein and membrane.
    labels : sequence of str, optional
        One per simulation, used in plots and messages.
    frame_step, start_frame, end_frame :
        Frame selection, defaulting to the configuration cell.
    """

    TOPOLOGY_SUFFIXES = {".gro", ".pdb", ".tpr"}
    TRAJECTORY_SUFFIXES = {".xtc", ".trr", ".dcd"}

    def __init__(self, topologies, trajectories, labels=None,
                 frame_step=None, start_frame=None, end_frame=None):
        # Accept bare paths as one-element lists.
        if isinstance(topologies, (str, Path)):
            topologies = [topologies]
        if isinstance(trajectories, (str, Path)):
            trajectories = [trajectories]
        self.topologies = [Path(p) for p in topologies]
        self.trajectories = [Path(p) for p in trajectories]

        if not self.topologies:
            raise ValueError("No topologies given.")
        if len(self.topologies) != len(self.trajectories):
            raise ValueError(
                f"Got {len(self.topologies)} topologies but "
                f"{len(self.trajectories)} trajectories; they must correspond.")

        for paths, suffixes, what in (
            (self.topologies, self.TOPOLOGY_SUFFIXES, "topology"),
            (self.trajectories, self.TRAJECTORY_SUFFIXES, "trajectory"),
        ):
            bad = [p for p in paths if p.suffix.lower() not in suffixes]
            if bad:
                raise ValueError(
                    f"Unsupported {what} type '{bad[0].suffix}'. "
                    f"Supported: {sorted(suffixes)}")

        missing = [p for p in self.topologies + self.trajectories if not p.is_file()]
        if missing:
            raise FileNotFoundError(
                f"{len(missing)} input file(s) not found. First: {missing[0]}\n"
                "Check DATA_ROOT and build_path() in the configuration cell.")

        self.labels = (list(labels) if labels is not None
                       else [p.parent.name for p in self.topologies])
        self.frame_step = FRAME_STEP if frame_step is None else frame_step
        self.start_frame = START_FRAME if start_frame is None else start_frame
        self.end_frame = END_FRAME if end_frame is None else end_frame

        self.universes = [mda.Universe(str(t), str(x))
                          for t, x in zip(self.topologies, self.trajectories)]

    # -- helpers ------------------------------------------------------------
    @staticmethod
    def _resname_selection(lipid):
        """Turn a resname, or a collection of them, into an MDAnalysis selection."""
        if isinstance(lipid, str):
            resnames = lipid.split()
        elif isinstance(lipid, (list, tuple, set, frozenset, np.ndarray)):
            resnames = [str(r) for r in lipid]
        else:
            raise TypeError(
                f"lipid must be a resname string or a collection of them, "
                f"got {type(lipid).__name__}")
        if not resnames:
            raise ValueError("lipid must name at least one resname.")
        return "resname " + " ".join(resnames)

    def _frame_bounds(self, universe):
        """Resolve the stop frame for one universe.

        Resolved per universe rather than once: the original code assigned the
        first trajectory's length to `end_frame` and then reused that value for
        every later universe, silently truncating longer runs.
        """
        stop = len(universe.trajectory) if self.end_frame is None else self.end_frame
        return self.start_frame, stop

    def _select(self, universe, protein_sel, lipid_sel, heavy_atoms_only):
        protein = universe.select_atoms(protein_sel)
        lipids = universe.select_atoms(lipid_sel)
        if heavy_atoms_only:
            # "not name H*" is the resolution-independent way to drop hydrogens
            # when the topology carries no element information.
            protein = protein.select_atoms("not name H*")
            lipids = lipids.select_atoms("not name H*")
        return protein, lipids

    # -- minimum distances --------------------------------------------------
    def lipid_distance(self, lipid, protein_sel="protein", heavy_atoms_only=True,
                       max_distance=None):
        """Minimum distance from each lipid molecule to the protein, per frame.

        Sets `self.min_distances`: a list with one (n_frames, n_lipids) array per
        simulation, in angstrom. Also sets `self.pooled_distances`, the same data
        flattened, which is what the distribution plots consume.

        `max_distance` caps the search: distances beyond it are recorded as
        infinity. That is a large speed-up when only close contacts matter, but
        it makes the pooled distribution incomplete, so the plotting helpers
        below drop non-finite values and report how many were dropped.
        """
        lipid_sel = self._resname_selection(lipid)
        max_distance = MAX_DISTANCE if max_distance is None else max_distance
        upper_bound = np.inf if max_distance is None else float(max_distance)
        if upper_bound <= 0:
            raise ValueError("max_distance must be positive.")

        self.min_distances = []
        for universe, label in zip(self.universes, self.labels):
            protein, lipids = self._select(universe, protein_sel, lipid_sel,
                                           heavy_atoms_only)
            start, stop = self._frame_bounds(universe)
            n_frames = len(range(start, stop, self.frame_step))
            lipid_residues = lipids.residues
            n_lipids = len(lipid_residues)

            if len(protein) == 0:
                raise ValueError(
                    f"{label}: the selection '{protein_sel}' matched no atoms. "
                    "Set PROTEIN_SELECTION for this model level.")
            if n_lipids == 0:
                raise ValueError(
                    f"{label}: the selection '{lipid_sel}' matched no atoms. "
                    "Check LIPID_RESNAMES against the resnames in your topology.")
            if n_frames == 0:
                raise ValueError(
                    f"{label}: frame selection [{start}:{stop}:{self.frame_step}] "
                    "is empty.")

            # Map each lipid atom to its molecule, then reduce atom distances to
            # a per-molecule minimum with np.minimum.reduceat.
            atom_to_molecule = np.searchsorted(lipid_residues.resindices,
                                               lipids.resindices)
            order = (None if np.all(np.diff(atom_to_molecule) >= 0)
                     else np.argsort(atom_to_molecule, kind="stable"))
            sorted_molecule = (atom_to_molecule if order is None
                               else atom_to_molecule[order])
            group_starts = np.flatnonzero(np.concatenate(
                ([True], sorted_molecule[1:] != sorted_molecule[:-1])))
            group_ids = sorted_molecule[group_starts]

            distances = np.full((n_frames, n_lipids), np.inf, dtype=np.float32)
            atom_min = np.empty(len(lipids), dtype=np.float64)
            chunk_size = max(1, int(8_000_000 // len(protein)))
            buffer = None

            for iframe, ts in enumerate(
                    universe.trajectory[start:stop:self.frame_step]):
                box = ts.dimensions
                has_box = box is not None and np.all(box[:3] > 0)
                orthorhombic = has_box and np.allclose(box[3:], 90.0)

                if orthorhombic or not has_box:
                    protein_pos = protein.positions.astype(np.float64, copy=True)
                    lipid_pos = lipids.positions.astype(np.float64, copy=True)
                    if orthorhombic:
                        # cKDTree's boxsize needs every coordinate strictly
                        # inside [0, L); wrap, then nudge off the upper edge.
                        lengths = box[:3].astype(np.float64)
                        np.mod(protein_pos, lengths, out=protein_pos)
                        np.mod(lipid_pos, lengths, out=lipid_pos)
                        edge = np.nextafter(lengths, 0.0)
                        np.clip(protein_pos, 0.0, edge, out=protein_pos)
                        np.clip(lipid_pos, 0.0, edge, out=lipid_pos)
                        tree = cKDTree(protein_pos, boxsize=lengths)
                    else:
                        tree = cKDTree(protein_pos)
                    atom_min[:], _ = tree.query(
                        lipid_pos, k=1, distance_upper_bound=upper_bound, workers=-1)
                else:
                    # Triclinic box: a KD-tree cannot express the periodicity,
                    # so fall back to MDAnalysis, chunked to bound memory.
                    if buffer is None:
                        buffer = np.empty((chunk_size, len(protein)), dtype=np.float64)
                    protein_pos = protein.positions
                    lipid_pos = lipids.positions
                    for begin in range(0, len(lipids), chunk_size):
                        block = lipid_pos[begin:begin + chunk_size]
                        result = buffer[:len(block)]
                        distance_array(block, protein_pos, box=box, result=result)
                        atom_min[begin:begin + len(block)] = result.min(axis=1)
                    if max_distance is not None:
                        atom_min[atom_min > upper_bound] = np.inf

                ordered = atom_min if order is None else atom_min[order]
                distances[iframe, group_ids] = np.minimum.reduceat(
                    ordered, group_starts)

            self.min_distances.append(distances)

        self.pooled_distances = [np.ravel(d) for d in self.min_distances]
        return self

    # -- residue contacts ---------------------------------------------------
    def lipid_residue_contacts(self, lipid, protein_sel="protein",
                               contact_threshold=5.0, heavy_atoms_only=True):
        """Boolean contact map per (frame, protein residue, lipid molecule).

        Sets `self.residue_contacts`: one (n_frames, n_residues, n_lipids) bool
        array per simulation. `capped_distance` returns only the pairs within the
        cutoff, so cost scales with the number of contacts rather than with
        n_protein x n_lipid.
        """
        if contact_threshold <= 0:
            raise ValueError("contact_threshold must be positive.")
        lipid_sel = self._resname_selection(lipid)

        self.residue_contacts = []
        for universe, label in zip(self.universes, self.labels):
            protein, lipids = self._select(universe, protein_sel, lipid_sel,
                                           heavy_atoms_only)
            start, stop = self._frame_bounds(universe)
            n_frames = len(range(start, stop, self.frame_step))
            protein_residues = protein.residues
            lipid_residues = lipids.residues
            n_res = len(protein_residues)
            n_lipids = len(lipid_residues)

            if n_res == 0 or n_lipids == 0 or n_frames == 0:
                raise ValueError(
                    f"{label}: empty selection or frame range "
                    f"({n_res} residues, {n_lipids} lipids, {n_frames} frames).")

            protein_atom_to_res = np.searchsorted(protein_residues.resindices,
                                                  protein.resindices)
            lipid_atom_to_res = np.searchsorted(lipid_residues.resindices,
                                                lipids.resindices)

            contacts = np.zeros((n_frames, n_res, n_lipids), dtype=bool)
            for iframe, ts in enumerate(
                    universe.trajectory[start:stop:self.frame_step]):
                box = ts.dimensions
                if box is None or not np.all(box[:3] > 0):
                    box = None
                pairs = capped_distance(
                    protein.positions, lipids.positions,
                    max_cutoff=contact_threshold, box=box, return_distances=False)
                if len(pairs) == 0:
                    continue
                contacts[iframe,
                         protein_atom_to_res[pairs[:, 0]],
                         lipid_atom_to_res[pairs[:, 1]]] = True

            self.residue_contacts.append(contacts)
        return self

    def residue_numbers(self, index=0, protein_sel="protein",
                        heavy_atoms_only=True, residue_offset=0):
        """Real residue numbers for the contact map's residue axis."""
        protein = self.universes[index].select_atoms(protein_sel)
        if heavy_atoms_only:
            protein = protein.select_atoms("not name H*")
        return protein.residues.resids + residue_offset


def contact_probability_per_residue(contacts, n_protomers=None):
    """Collapse a contact map to a per-residue probability for one protomer.

    Takes a (n_frames, n_residues, n_lipids) boolean array and returns a
    1-D probability of length n_residues // n_protomers.

    A residue counts as "in contact" in a frame if it touches *any* molecule of
    the species, so this is occupancy, not a count of bound lipids. Protomers are
    averaged, as elsewhere in this repository.
    """
    n_protomers = N_PROTOMERS if n_protomers is None else n_protomers
    contacts = np.asarray(contacts)
    n_frames, n_res_total, _ = contacts.shape
    if n_res_total % n_protomers != 0:
        raise ValueError(
            f"{n_res_total} residues is not divisible by n_protomers="
            f"{n_protomers}. Either the protein selection did not cover whole "
            "subunits, or N_PROTOMERS is wrong.")
    per_frame = contacts.any(axis=2)                      # (frames, residues)
    per_protomer = per_frame.reshape(n_frames, n_protomers, -1)
    return per_protomer.mean(axis=(0, 1))

---

## 6. Lipid plotting

In [ ]:
# ===========================================================================
#  Lipid-specific plotting
# ===========================================================================

def _finite(values, context=""):
    """Drop non-finite entries, reporting how many were removed.

    Distances come back as infinity when MAX_DISTANCE excluded a pair. Silently
    dropping them would misrepresent the distribution, so the count is printed.
    """
    values = np.asarray(values, dtype=float).ravel()
    mask = np.isfinite(values)
    n_dropped = int(values.size - mask.sum())
    if n_dropped:
        print(f"  note: dropped {n_dropped} non-finite value(s){context} "
              "(these were beyond MAX_DISTANCE)")
    return values[mask]


def plot_distance_histogram(distances, label, color=None, bin_width=0.3,
                            x_max=None, figure_name=None):
    """Probability density of lipid-protein minimum distance for one species.

    The first peak is the contact shell: its position is how closely this species
    packs against the protein, and its area is how much of the species is bound
    at any moment.
    """
    values = _finite(distances, f" for {label}")
    if values.size == 0:
        print(f"  {label}: no finite distances to plot")
        return None, None

    x_max = float(np.percentile(values, 99.5)) if x_max is None else x_max
    bins = np.arange(0.0, x_max + bin_width, bin_width)

    fig, ax = plt.subplots(figsize=(15, 8))
    ax.hist(values, bins=bins, density=True,
            color=color or LIPID_COLORS.get(label, "#444444"),
            edgecolor="black", linewidth=0.5)
    ax.set_xlabel(r"Lipid–Protein Distance ($\mathrm{\AA}$)")
    ax.set_ylabel("Probability Density")
    ax.set_title(label, fontsize=FONTSIZE)
    ax.set_xlim(0.0, x_max)
    apply_ticks(ax, "x", 0.0, x_max, n_ticks=9)
    lo, hi = ax.get_ylim()
    apply_ticks(ax, "y", lo, hi, n_ticks=6)

    if figure_name:
        save_figure(fig, figure_name)
    return fig, ax


def plot_distance_violins(distances_by_label, labels, colors=None, y_max=None,
                          title=None, figure_name=None):
    """Compare the distance distributions of every lipid species side by side.

    A violin whose mass sits low against the axis is a species that stays in
    contact; one centred well away from zero is bulk membrane.
    """
    records = []
    for values, label in zip(distances_by_label, labels):
        for value in _finite(values, f" for {label}"):
            records.append({"Lipid": label, "Distance": value})
    if not records:
        print("  no finite distances to plot")
        return None, None
    frame = pd.DataFrame.from_records(records)

    colors = (colors if colors is not None
              else [LIPID_COLORS.get(l, "#444444") for l in labels])

    fig, ax = plt.subplots(figsize=(15, 8))
    sns.violinplot(data=frame, x="Lipid", y="Distance", hue="Lipid",
                   palette=list(colors), legend=False, ax=ax,
                   cut=0, inner="quartile", bw_adjust=0.3, density_norm="area")
    ax.set_xlabel("Lipid Species")
    ax.set_ylabel(r"Lipid–Protein Distance ($\mathrm{\AA}$)")
    if title:
        ax.set_title(title, fontsize=FONTSIZE)
    y_max = (float(frame["Distance"].quantile(0.995)) if y_max is None else y_max)
    ax.set_ylim(0.0, y_max)
    apply_ticks(ax, "y", 0.0, y_max, n_ticks=8)

    if figure_name:
        save_figure(fig, figure_name)
    return fig, ax


def plot_residue_contact_probability(residues, probabilities, labels, colors=None,
                                     legend_title="Lipid species", title=None,
                                     segments=None, show_domain_legend=False,
                                     figure_name=None):
    """Per-residue contact probability, one line per lipid species.

    Peaks are candidate lipid binding sites. Species are drawn with the least
    contacted in front, so a low-occupancy species is not hidden behind a
    high-occupancy one.
    """
    lengths = {len(p) for p in probabilities} | {len(residues)}
    if len(lengths) != 1:
        raise ValueError(
            f"Probability profiles and the residue axis have differing lengths "
            f"{sorted(lengths)}.")
    residues = np.asarray(residues)
    colors = (colors if colors is not None
              else [LIPID_COLORS.get(l, "#444444") for l in labels])

    fig, ax = plt.subplots(figsize=(15, 8))

    # Draw the most-contacted species first so it ends up behind the others.
    means = np.array([np.mean(p) for p in probabilities])
    draw_order = np.argsort(-means)
    for rank, i in enumerate(draw_order):
        ax.plot(residues, probabilities[i], color=colors[i], label=labels[i],
                linewidth=3, zorder=3 + rank)

    ax.set_xlabel("Residue Number")
    ax.set_ylabel("Contact Probability")
    if title:
        ax.set_title(title, fontsize=FONTSIZE)
    ax.set_xlim(residues[0], residues[-1])
    apply_ticks(ax, "x", float(residues[0]), float(residues[-1]), n_ticks=9)
    ax.set_ylim(0.0, 1.0)
    apply_ticks(ax, "y", 0.0, 1.0, n_ticks=6)

    # Order the legend by occupancy, which is the order a reader cares about.
    handles, legend_labels = ax.get_legend_handles_labels()
    by_label = dict(zip(legend_labels, handles))
    ordered = [labels[i] for i in draw_order]
    ax.legend([by_label[l] for l in ordered], ordered, title=legend_title,
              loc="upper left", ncol=2, fontsize=FONTSIZE * 0.7,
              title_fontsize=FONTSIZE * 0.7)

    add_domain_bands(ax, segments=segments, show_legend=show_domain_legend)

    if figure_name:
        save_figure(fig, figure_name)
    return fig, ax

---

# Analysis

Everything above defines the pipeline; everything below runs it.

---

## 7. Compute distances and contacts

In [ ]:
# ===========================================================================
#  Run the analysis
#
#  One pass per (model level, lipid species). The original notebook repeated a
#  near-identical eight-line block for every species and every resolution -
#  fourteen copies, each with its own hard-coded index. Looping over the
#  configuration means adding a lipid species is a one-line change.
# ===========================================================================

lipid_data = {}

for model_level in MODEL_LEVELS:
    protein_sel = PROTEIN_SELECTION[model_level]
    heavy_only = HEAVY_ATOMS_ONLY[model_level]
    threshold = CONTACT_THRESHOLD[model_level]
    print(f"\n{model_level} (cutoff {threshold} A, "
          f"heavy atoms only: {heavy_only}, selection '{protein_sel}')")

    topologies = grid.paths(model_level, "topology", analysis_indices)
    trajectories = grid.paths(model_level, "trajectory", analysis_indices)
    labels = grid.labels(analysis_indices, by="both")

    per_lipid = {}
    for lipid_label, resnames in LIPID_RESNAMES.items():
        resname = resnames[model_level]
        analysis = LipidAnalysis(topologies, trajectories, labels=labels)
        analysis.lipid_distance(resname, protein_sel=protein_sel,
                                heavy_atoms_only=heavy_only)
        analysis.lipid_residue_contacts(resname, protein_sel=protein_sel,
                                        contact_threshold=threshold,
                                        heavy_atoms_only=heavy_only)
        per_lipid[lipid_label] = analysis

        n_lipids = analysis.min_distances[0].shape[1]
        closest = np.nanmin(analysis.min_distances[0])
        print(f"  {lipid_label:<6s} resname {resname:<5s} "
              f"{n_lipids:4d} molecules   closest approach {closest:6.2f} A")

    residues = per_lipid[next(iter(per_lipid))].residue_numbers(
        0, protein_sel=protein_sel, heavy_atoms_only=heavy_only,
        residue_offset=RESIDUE_OFFSET.get(model_level, 0))

    lipid_data[model_level] = {"per_lipid": per_lipid, "residues": residues}

print("\nDone.")

---

## 8. Distance summary

In [ ]:
# Which species sits closest to the channel?
#
# The median is the more robust summary; the 5th percentile says how close the
# tightest-bound molecules get. "Bound fraction" is the proportion of
# (molecule, frame) observations inside the contact cutoff, i.e. how much of the
# species is engaged with the protein at any instant.
for model_level, entry in lipid_data.items():
    threshold = CONTACT_THRESHOLD[model_level]
    print(f"\n{'=' * 74}\n{model_level.upper()}  (contact cutoff {threshold} A)"
          f"\n{'=' * 74}")
    print(f"{'lipid':<8}{'median (A)':>12}{'5th pct (A)':>14}"
          f"{'min (A)':>10}{'bound fraction':>17}")
    print("-" * 74)

    rows = []
    for label, analysis in entry["per_lipid"].items():
        values = np.asarray(analysis.pooled_distances[0], dtype=float)
        values = values[np.isfinite(values)]
        if values.size == 0:
            continue
        rows.append((label, np.median(values), np.percentile(values, 5),
                     values.min(), float(np.mean(values <= threshold))))

    for label, median, p5, minimum, bound in sorted(rows, key=lambda r: r[1]):
        print(f"{label:<8}{median:>12.2f}{p5:>14.2f}{minimum:>10.2f}"
              f"{bound:>17.3f}")

---

## 9. Distance distributions

In [ ]:
# Distance distributions. The violin plot is the summary figure; the individual
# histograms resolve the shape of the contact shell for one species at a time.
SHOW_INDIVIDUAL_HISTOGRAMS = False

for model_level, entry in lipid_data.items():
    labels = list(entry["per_lipid"])
    pooled = [entry["per_lipid"][l].pooled_distances[0] for l in labels]

    plot_distance_violins(
        pooled, labels,
        title=f"{model_level.title()}: lipid–protein distance distributions",
        figure_name=f"lipid_distance_violins_{model_level}",
    )
    plt.show()

    if SHOW_INDIVIDUAL_HISTOGRAMS:
        for label in labels:
            plot_distance_histogram(
                entry["per_lipid"][label].pooled_distances[0], label,
                figure_name=f"lipid_distance_hist_{model_level}_{label}",
            )
            plt.show()

---

## 10. Residue contact profiles

In [ ]:
# Per-residue contact probability, one line per lipid species. Peaks are the
# candidate binding sites.
for model_level, entry in lipid_data.items():
    labels = list(entry["per_lipid"])
    profiles = [
        contact_probability_per_residue(
            entry["per_lipid"][l].residue_contacts[0]) for l in labels
    ]
    residues = entry["residues"]
    # The residue axis covers the whole assembly; the profiles cover one
    # protomer, so take the first protomer's slice of residue numbers.
    residues_protomer = residues[:len(profiles[0])]

    plot_residue_contact_probability(
        residues=residues_protomer,
        probabilities=profiles,
        labels=labels,
        title=f"{model_level.title()}: lipid contact probability per residue",
        figure_name=f"lipid_residue_contacts_{model_level}",
    )
    plt.show()

---

## 11. Candidate binding sites

In [ ]:
# Candidate lipid binding sites: residues contacted far more often than that
# species' own average across the channel.
#
# The enrichment is relative to each species separately, so an abundant lipid
# that touches everything does not swamp a rare one with a specific site.
ENRICHMENT_FACTOR = 3.0      # times the species' mean occupancy
MIN_OCCUPANCY = 0.15         # ignore sites that are rarely occupied at all

for model_level, entry in lipid_data.items():
    print(f"\n{'=' * 78}\n{model_level.upper()}  "
          f"(enrichment > {ENRICHMENT_FACTOR}x mean, occupancy > {MIN_OCCUPANCY})"
          f"\n{'=' * 78}")
    residues = entry["residues"]
    any_found = False

    for label, analysis in entry["per_lipid"].items():
        profile = contact_probability_per_residue(analysis.residue_contacts[0])
        residues_protomer = residues[:len(profile)]
        mean_occupancy = float(np.mean(profile))
        if mean_occupancy <= 0:
            continue
        threshold = max(ENRICHMENT_FACTOR * mean_occupancy, MIN_OCCUPANCY)
        hits = np.flatnonzero(profile >= threshold)
        if hits.size == 0:
            continue

        any_found = True
        print(f"\n{label}  (mean occupancy {mean_occupancy:.3f})")
        print(f"  {'residue':>9}{'occupancy':>12}{'enrichment':>13}   domain")
        for index in hits:
            residue = int(residues_protomer[index])
            print(f"  {residue:>9}{profile[index]:>12.3f}"
                  f"{profile[index] / mean_occupancy:>12.1f}x   "
                  f"{describe_domain(residue)}")

    if not any_found:
        print("\nNo residues passed the enrichment threshold. Either the lipids "
              "interact non-specifically, or the thresholds above are too strict.")

---

## Troubleshooting

**`the selection '...' matched no atoms`**
For the protein, MDAnalysis' `"protein"` keyword relies on standard amino-acid
resnames; many coarse-grained topologies do not use them. Set
`PROTEIN_SELECTION[model_level]` to an explicit selection such as
`"name BB SC1 SC2 SC3 SC4"`. Check what is actually present:

```python
u = mda.Universe(str(grid.paths("coarse", "topology", analysis_indices)[0]))
print(sorted(set(u.residues.resnames)))
print(sorted(set(u.atoms.names))[:40])
```

For lipids, the resname in `LIPID_RESNAMES` does not match the topology. The
same snippet lists the resnames available.

**`N residues is not divisible by n_protomers`**
The protein selection did not cover whole subunits, or `N_PROTOMERS` is wrong.

**Everything is at infinity, or the distributions look truncated**
`MAX_DISTANCE` is set, so distances beyond it were never computed. Set it to
`None` for exact distances everywhere.

**It is far too slow**
Cost scales as frames x lipid molecules x protein atoms. In order of
effectiveness: raise `FRAME_STEP`; analyse fewer entries in
`ANALYSIS_CONDITIONS`; set `MAX_DISTANCE` to a value a little above the contact
cutoff; and for atomistic systems keep `HEAVY_ATOMS_ONLY` True.

**Contact probabilities are all near 1**
The cutoff is too generous for the resolution — check `CONTACT_THRESHOLD`.
A coarse-grained bead needs roughly 6 Å, an all-atom heavy atom roughly 4.5 Å.

---

## Notes on method

- **Contact means occupancy, not stoichiometry.** A residue is "in contact" in a
  frame if *any* molecule of the species is within the cutoff. Two cholesterols
  at one site read the same as one.
- **Protomer averaging** assumes a homo-oligomer whose subunits appear as
  consecutive equal-length residue blocks. Set `N_PROTOMERS = 1` for a
  hetero-oligomer and slice the profiles yourself.
- **Coarse and atomistic numbers are not directly comparable.** Different
  cutoffs, different particle definitions, and a coarse bead's centre is not an
  atom position. Compare trends between them, not absolute probabilities.
- **The distance distribution is over molecules and frames pooled together.**
  It therefore mixes "many lipids sometimes close" with "few lipids always
  close". The per-residue profile is what separates those two cases.
- **Enrichment is computed per species**, relative to that species' own mean
  occupancy across the channel, so abundance does not create false sites. It is
  a screening heuristic: confirm any hit by looking at the structure.
- **Periodic boundaries** are applied throughout. For orthorhombic boxes a
  periodic KD-tree is used; triclinic boxes fall back to MDAnalysis'
  `distance_array`, which is slower but handles the general case.

---

## Repository housekeeping

This notebook is one of several in the repository, so a few conventions keep
them from interfering with each other.

**Shared, repository-level files.** `requirements.txt` and `.gitignore` live at
the repository root, one of each, covering every notebook. Neither can be
renamed per notebook: git only recognises the filename `.gitignore`, and
`pip install -r requirements.txt` is what every reader and CI runner expects.

**Per-notebook outputs.** Each notebook writes to `results/<NOTEBOOK_SLUG>/`,
set in the configuration cell, so two notebooks saving a figure of the same name
cannot overwrite one another.

**Shared input data.** `DATA_ROOT` and `demo_data/` are deliberately *not*
namespaced, so sibling notebooks read the same trajectories rather than each
keeping a copy.

**Sections 1-4 of the configuration cell are identical in every notebook.** If
you change the simulation grid, the protomer count or the domain boundaries,
change them the same way in each notebook. If that becomes tedious, move those
sections into a `channel_config.py` at the repository root and import it from
each notebook instead.

**Outputs are stripped** from the committed notebooks, which keeps diffs small
and stops absolute data paths leaking through cell output:

```bash
pip install nbstripout
nbstripout --install     # run once, inside the repository
```
